# 11 — Kaggle VSL: MediaPipe keypoints → graph cache

Notebook này **không chạy lại RTMPose**. Nó tải đúng thư mục keypoint MediaPipe đã được dataset công bố, sao chép toàn bộ `.npy` sang Google Drive, sau đó chuyển tập con được chọn thành tensor graph `[64, 75, 7]`.

Luồng giữ nguyên phần học: **Graph Encoder → Spatial Transformer → Temporal Transformer**. Chỉ extractor/layout đầu vào đổi từ COCO-WholeBody sang `33 pose + 21 tay trái + 21 tay phải`. Điểm thứ 76 chưa được data card định nghĩa nên không được dùng trong graph v1; file nguồn trên Drive vẫn giữ nguyên đầy đủ 76 điểm.

> Split công khai chỉ có train/test và manifest xử lý không công bố signer ID. Notebook giữ nguyên test, rồi tách validation có phân tầng từ train. Không trình bày validation này là signer-disjoint.

In [ ]:
#@title Cấu hình
PROJECT_GIT_REF = 'feat/kaggle-vsl-mediapipe-graph'  #@param {type:'string'}
DATASET_HANDLE = 'nguyenanfms/vsl-vietnamese-sign-language-v2/versions/8'  #@param {type:'string'}
KAGGLE_KEYPOINT_DIRECTORY = 'processed/processed/keypoints_splited'  #@param {type:'string'}
CLASS_COUNT = 50  #@param {type:'integer'}
MIN_OFFICIAL_TRAIN_SAMPLES = 40  #@param {type:'integer'}
VALIDATION_FRACTION = 0.20  #@param {type:'number'}
SEED = 42  #@param {type:'integer'}
COPY_ALL_KEYPOINTS_TO_DRIVE = True  #@param {type:'boolean'}
CONVERT_LIMIT = 0  #@param {type:'integer'}
CONTINUE_ON_ERROR = False  #@param {type:'boolean'}

assert CLASS_COUNT >= 0
assert MIN_OFFICIAL_TRAIN_SAMPLES >= 2
assert 0 < VALIDATION_FRACTION < 1


In [ ]:
#@title Mount Google Drive và khai báo đường dẫn
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/silent-signal-results/vsl_kaggle_mediapipe')
DRIVE_KEYPOINT_ROOT = DRIVE_ROOT / 'source/keypoints_splited'
SUBSET_ROOT = DRIVE_ROOT / f'subsets/top{CLASS_COUNT}'
MANIFEST = SUBSET_ROOT / 'manifest.csv'
LABELS = SUBSET_ROOT / 'labels.json'
SELECTION = SUBSET_ROOT / 'selection.json'
BUILD_REPORT = SUBSET_ROOT / 'manifest_report.json'
PINNED_GRAPH_CONFIG = SUBSET_ROOT / 'mediapipe_graph_config.pinned.yaml'
GRAPH_ROOT = SUBSET_ROOT / 'graph/mediapipe_holistic_75_v1_t64/cache'
GRAPH_REPORT = SUBSET_ROOT / 'graph/mediapipe_holistic_75_v1_t64/report.json'
LOCAL_DOWNLOAD_ROOT = Path('/content/kaggle-vsl-keypoints')
LOCAL_REPO = Path('/content/silent-signal')

for path in (DRIVE_ROOT, DRIVE_KEYPOINT_ROOT, SUBSET_ROOT, GRAPH_ROOT):
    path.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE_ROOT)


In [ ]:
#@title Cài KaggleHub và tải riêng thư mục keypoint vào ổ Colab
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub>=1.0.2'], check=True)

import kagglehub
LOCAL_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
downloaded = Path(kagglehub.dataset_download(
    DATASET_HANDLE,
    path=KAGGLE_KEYPOINT_DIRECTORY,
    output_dir=str(LOCAL_DOWNLOAD_ROOT),
    force_download=False,
))
LOCAL_KEYPOINT_ROOT = downloaded
npy_count = sum(1 for _ in LOCAL_KEYPOINT_ROOT.rglob('*.npy'))
if npy_count == 0:
    raise RuntimeError(f'Không tìm thấy .npy trong {LOCAL_KEYPOINT_ROOT}')
print('Downloaded directory:', LOCAL_KEYPOINT_ROOT)
print('Keypoint files:', f'{npy_count:,}')


## Sao chép toàn bộ keypoint đã extract sang Drive

Cell dưới đây là cell lưu **toàn bộ** `.npy` MediaPipe vào Drive. Có thể chạy lại: file đích đã tồn tại và có cùng dung lượng sẽ được bỏ qua. Việc chép nhiều file nhỏ qua Google Drive có thể lâu; không đóng runtime giữa chừng.

In [ ]:
#@title Copy toàn bộ .npy sang Drive — có resume
import json, shutil, time

if COPY_ALL_KEYPOINTS_TO_DRIVE:
    source_files = sorted(LOCAL_KEYPOINT_ROOT.rglob('*.npy'))
    copied = skipped = 0
    copied_bytes = 0
    started = time.perf_counter()
    for position, source in enumerate(source_files, start=1):
        relative = source.relative_to(LOCAL_KEYPOINT_ROOT)
        destination = DRIVE_KEYPOINT_ROOT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.is_file() and destination.stat().st_size == source.stat().st_size:
            skipped += 1
        else:
            shutil.copy2(source, destination)
            copied += 1
            copied_bytes += source.stat().st_size
        if position == 1 or position == len(source_files) or position % 500 == 0:
            elapsed = time.perf_counter() - started
            rate = position / elapsed if elapsed else 0
            eta = (len(source_files) - position) / rate / 60 if rate else 0
            print(f'{position:,}/{len(source_files):,} | copied={copied:,} | skipped={skipped:,} | ETA={eta:.1f} min')
    persisted = sum(1 for _ in DRIVE_KEYPOINT_ROOT.rglob('*.npy'))
    marker = {
        'dataset_handle': DATASET_HANDLE,
        'source_directory': KAGGLE_KEYPOINT_DIRECTORY,
        'source_files': len(source_files),
        'persisted_files': persisted,
        'complete': persisted == len(source_files),
    }
    (DRIVE_KEYPOINT_ROOT / '_copy_report.json').write_text(
        json.dumps(marker, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    if not marker['complete']:
        raise RuntimeError(f'Copy chưa đủ: {marker}')
    print(json.dumps(marker, ensure_ascii=False, indent=2))
else:
    print('Bỏ qua lưu raw keypoint lên Drive theo cấu hình.')


In [ ]:
#@title Checkout code và cài project
import subprocess, sys

if not (LOCAL_REPO / '.git').is_dir():
    subprocess.run([
        'git', 'clone', '--branch', PROJECT_GIT_REF, '--single-branch',
        'https://github.com/stillthethrone/silent-signal.git', str(LOCAL_REPO),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(LOCAL_REPO), 'fetch', 'origin', PROJECT_GIT_REF], check=True)
    subprocess.run(['git', '-C', str(LOCAL_REPO), 'checkout', '-B', PROJECT_GIT_REF, f'origin/{PROJECT_GIT_REF}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(LOCAL_REPO)], check=True)
print('Commit:', subprocess.check_output(['git', '-C', str(LOCAL_REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
#@title Chọn lớp, giữ test và tạo validation
import subprocess, sys

# Dùng bản local để quét nhanh; Drive đã giữ bản sao bền vững ở cell trên.
KEYPOINT_ROOT_FOR_CONVERSION = LOCAL_KEYPOINT_ROOT
command = [
    sys.executable, '-m', 'silent_signal.cli.prepare_kaggle_vsl_mediapipe', 'build',
    '--keypoint-root', str(KEYPOINT_ROOT_FOR_CONVERSION),
    '--manifest', str(MANIFEST),
    '--labels', str(LABELS),
    '--selection', str(SELECTION),
    '--report', str(BUILD_REPORT),
    '--classes', str(CLASS_COUNT),
    '--min-official-train-samples', str(MIN_OFFICIAL_TRAIN_SAMPLES),
    '--validation-fraction', str(VALIDATION_FRACTION),
    '--seed', str(SEED),
    '--dataset-handle', DATASET_HANDLE,
]
print('+', ' '.join(command))
subprocess.run(command, check=True)


In [ ]:
#@title Pin config bằng manifest vừa tạo
import hashlib, json, yaml
from collections import Counter
import csv

payload = yaml.safe_load((LOCAL_REPO / 'configs/preprocessing/mediapipe_holistic_75_t64.yaml').read_text())
manifest_sha = hashlib.sha256(MANIFEST.read_bytes()).hexdigest()
with MANIFEST.open(encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
payload['expected'] = {
    'manifest_sha256': manifest_sha,
    'clips': len(rows),
    'classes': len({int(row['class_index']) for row in rows}),
    'splits': dict(Counter(row['split'] for row in rows)),
}
PINNED_GRAPH_CONFIG.write_text(yaml.safe_dump(payload, sort_keys=False), encoding='utf-8')
print(PINNED_GRAPH_CONFIG.read_text())


In [ ]:
#@title Chuyển MediaPipe .npy thành graph cache — có resume
import subprocess, sys

command = [
    sys.executable, '-u', '-m', 'silent_signal.cli.prepare_kaggle_vsl_mediapipe', 'convert',
    '--keypoint-root', str(KEYPOINT_ROOT_FOR_CONVERSION),
    '--manifest', str(MANIFEST),
    '--config', str(PINNED_GRAPH_CONFIG),
    '--output-root', str(GRAPH_ROOT),
    '--report', str(GRAPH_REPORT),
    '--progress-every', '50',
]
if CONVERT_LIMIT > 0:
    command += ['--limit', str(CONVERT_LIMIT)]
if CONTINUE_ON_ERROR:
    command.append('--continue-on-error')
print('+', ' '.join(command))
subprocess.run(command, check=True)


In [ ]:
#@title Kiểm tra kết quả cuối
import json, numpy as np

report = json.loads(GRAPH_REPORT.read_text(encoding='utf-8'))
cache_files = sorted(GRAPH_ROOT.rglob('*.npz'))
if not cache_files:
    raise RuntimeError('Không có graph cache nào được tạo.')
with np.load(cache_files[0], allow_pickle=False) as sample:
    print('Sample cache:', cache_files[0])
    print('features:', sample['features'].shape)
    print('joint_mask:', sample['joint_mask'].shape)
    print('adjacency:', sample['adjacency'].shape)
print(json.dumps({key: value for key, value in report.items() if key != 'failures'}, indent=2))
print('Raw MediaPipe on Drive:', DRIVE_KEYPOINT_ROOT)
print('Graph cache on Drive:', GRAPH_ROOT)
print('Manifest:', MANIFEST)
print('Labels:', LABELS)


## Đầu ra

- `source/keypoints_splited/`: toàn bộ `.npy [T,76,3]` tải từ Kaggle và lưu bền vững trên Drive.
- `subsets/top50/manifest.csv`: nhãn liên tục, test công khai được giữ nguyên.
- `subsets/top50/selection.json`: lớp được chọn và số mẫu từng split.
- `graph/mediapipe_holistic_75_v1_t64/cache/`: tensor `[64,75,7]` cho Graph Encoder → Spatial Transformer → Temporal Transformer.

Notebook train chỉ cần đọc `manifest.csv`, `labels.json`, config đã pin và thư mục graph cache; không cần tải lại Kaggle hoặc chạy pose extractor.